#### Load and basic validation

In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")

datasets = {}

for file in RAW_DIR.glob("*.csv"):
    name = file.stem
    datasets[name] = pd.read_csv(file)

print("Raw datasets loaded:\n")

for name, df in datasets.items():
    print(f"{name:20} | Rows: {len(df):5} | Columns: {len(df.columns):2}")

Raw datasets loaded:

agents               | Rows:    30 | Columns: 10
claims               | Rows:   300 | Columns: 12
claim_assessments    | Rows:   257 | Columns:  8
claim_documents      | Rows:   601 | Columns:  6
claim_payments       | Rows:    68 | Columns:  9
contracts            | Rows:   250 | Columns: 10
customers            | Rows:   500 | Columns: 15
garages              | Rows:    30 | Columns:  6
hospitals            | Rows:    20 | Columns:  6
locations            | Rows:    20 | Columns:  6
payments             | Rows:   500 | Columns:  7
policies             | Rows:   750 | Columns: 14
policy_coverage      | Rows:  2249 | Columns:  7
premium_payments     | Rows:  1211 | Columns:  7
vehicles             | Rows:   500 | Columns: 12


#### Check missing values and duplicates

In [2]:
print("MISSING VALUES")
print("=" * 60)

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")

    if len(missing) == 0:
        print("  No missing values")
    else:
        print(missing)


print("\n\nDUPLICATE RECORDS")
print("=" * 60)

for name, df in datasets.items():
    print(
        f"{name:20} | "
        f"Duplicates: {df.duplicated().sum()}"
    )

MISSING VALUES

agents
  No missing values

claims
approved_amount    145
dtype: int64

claim_assessments
approved_amount    102
dtype: int64

claim_documents
  No missing values

claim_payments
garage_id      54
hospital_id    52
dtype: int64

contracts
  No missing values

customers
kyc_verified_date    61
dtype: int64

garages
  No missing values

hospitals
  No missing values

locations
  No missing values

payments
  No missing values

policies
  No missing values

policy_coverage
  No missing values

premium_payments
  No missing values

vehicles
engine_capacity_cc    64
dtype: int64


DUPLICATE RECORDS
agents               | Duplicates: 0
claims               | Duplicates: 0
claim_assessments    | Duplicates: 0
claim_documents      | Duplicates: 0
claim_payments       | Duplicates: 0
contracts            | Duplicates: 0
customers            | Duplicates: 0
garages              | Duplicates: 0
hospitals            | Duplicates: 0
locations            | Duplicates: 0
payments     

#### Referential integrity

In [3]:
checks = {}

checks["Customer → Location"] = (
    datasets["customers"]["location_id"]
    .isin(datasets["locations"]["location_id"])
).all()

checks["Agent → Location"] = (
    datasets["agents"]["location_id"]
    .isin(datasets["locations"]["location_id"])
).all()

checks["Vehicle → Customer"] = (
    datasets["vehicles"]["customer_id"]
    .isin(datasets["customers"]["customer_id"])
).all()

checks["Policy → Customer"] = (
    datasets["policies"]["customer_id"]
    .isin(datasets["customers"]["customer_id"])
).all()

checks["Policy → Vehicle"] = (
    datasets["policies"]["vehicle_id"]
    .isin(datasets["vehicles"]["vehicle_id"])
).all()

checks["Policy → Agent"] = (
    datasets["policies"]["agent_id"]
    .isin(datasets["agents"]["agent_id"])
).all()

checks["Coverage → Policy"] = (
    datasets["policy_coverage"]["policy_id"]
    .isin(datasets["policies"]["policy_id"])
).all()

checks["Premium Payment → Policy"] = (
    datasets["premium_payments"]["policy_id"]
    .isin(datasets["policies"]["policy_id"])
).all()

checks["Claim → Policy"] = (
    datasets["claims"]["policy_id"]
    .isin(datasets["policies"]["policy_id"])
).all()

checks["Claim → Vehicle"] = (
    datasets["claims"]["vehicle_id"]
    .isin(datasets["vehicles"]["vehicle_id"])
).all()

checks["Assessment → Claim"] = (
    datasets["claim_assessments"]["claim_id"]
    .isin(datasets["claims"]["claim_id"])
).all()

checks["Document → Claim"] = (
    datasets["claim_documents"]["claim_id"]
    .isin(datasets["claims"]["claim_id"])
).all()

checks["Claim Payment → Claim"] = (
    datasets["claim_payments"]["claim_id"]
    .isin(datasets["claims"]["claim_id"])
).all()

checks["Claim Payment → Garage"] = (
    datasets["claim_payments"]["garage_id"]
    .dropna()
    .isin(datasets["garages"]["garage_id"])
).all()

checks["Claim Payment → Hospital"] = (
    datasets["claim_payments"]["hospital_id"]
    .dropna()
    .isin(datasets["hospitals"]["hospital_id"])
).all()


for check, result in checks.items():
    print(f"{check:35} : {'PASS' if result else 'FAIL'}")

Customer → Location                 : PASS
Agent → Location                    : PASS
Vehicle → Customer                  : PASS
Policy → Customer                   : PASS
Policy → Vehicle                    : PASS
Policy → Agent                      : PASS
Coverage → Policy                   : PASS
Premium Payment → Policy            : PASS
Claim → Policy                      : PASS
Claim → Vehicle                     : PASS
Assessment → Claim                  : PASS
Document → Claim                    : PASS
Claim Payment → Claim               : PASS
Claim Payment → Garage              : PASS
Claim Payment → Hospital            : PASS


#### Business-rule checks

In [7]:
# ============================================================
# BUSINESS RULE VALIDATION
# ============================================================

policies = datasets["policies"].copy()
claims = datasets["claims"].copy()
customers = datasets["customers"].copy()

# Convert date columns after reading CSV
customers["date_of_birth"] = pd.to_datetime(
    customers["date_of_birth"],
    errors="coerce"
)

customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)

policies["issue_date"] = pd.to_datetime(
    policies["issue_date"],
    errors="coerce"
)

policies["start_date"] = pd.to_datetime(
    policies["start_date"],
    errors="coerce"
)

policies["end_date"] = pd.to_datetime(
    policies["end_date"],
    errors="coerce"
)

claims["claim_date"] = pd.to_datetime(
    claims["claim_date"],
    errors="coerce"
)

claims["incident_date"] = pd.to_datetime(
    claims["incident_date"],
    errors="coerce"
)


print("BUSINESS RULE VALIDATION")
print("=" * 60)

# ------------------------------------------------------------
# Policy checks
# ------------------------------------------------------------

print(
    "Policy end date > start date:",
    (
        policies["end_date"]
        > policies["start_date"]
    ).all()
)

print(
    "Policy issue date <= start date:",
    (
        policies["issue_date"]
        <= policies["start_date"]
    ).all()
)

print(
    "Policy premium > 0:",
    (
        policies["premium_amount"]
        > 0
    ).all()
)

print(
    "Policy sum insured > 0:",
    (
        policies["sum_insured"]
        > 0
    ).all()
)


# ------------------------------------------------------------
# Customer checks
# ------------------------------------------------------------

customer_age = (
    customers["registration_date"]
    - customers["date_of_birth"]
).dt.days / 365.25

print(
    "Customer age >= 18:",
    (
        customer_age >= 18
    ).all()
)


# ------------------------------------------------------------
# Claim checks
# ------------------------------------------------------------

print(
    "Claim date >= incident date:",
    (
        claims["claim_date"]
        >= claims["incident_date"]
    ).all()
)

print(
    "Claim estimated amount > 0:",
    (
        claims["estimated_amount"]
        > 0
    ).all()
)


approved_claims = claims[
    claims["approved_amount"].notna()
]

print(
    "Approved amount <= estimated amount:",
    (
        approved_claims["approved_amount"]
        <= approved_claims["estimated_amount"]
    ).all()
)

BUSINESS RULE VALIDATION
Policy end date > start date: True
Policy issue date <= start date: True
Policy premium > 0: True
Policy sum insured > 0: True
Customer age >= 18: True
Claim date >= incident date: True
Claim estimated amount > 0: True
Approved amount <= estimated amount: True


#### Quick business summary

In [6]:
print("INSURAFLOW RAW DATA SUMMARY")
print("=" * 60)

print(f"Customers        : {len(customers):,}")
print(f"Vehicles         : {len(datasets['vehicles']):,}")
print(f"Policies         : {len(policies):,}")
print(f"Claims           : {len(claims):,}")
print(f"Premium Payments : {len(datasets['premium_payments']):,}")
print(f"Claim Payments   : {len(datasets['claim_payments']):,}")

print("\nPolicy Status:")
print(policies["policy_status"].value_counts())

print("\nClaim Status:")
print(claims["claim_status"].value_counts())

print("\nClaim Type:")
print(claims["claim_type"].value_counts())

print("\nVehicle Type:")
print(datasets["vehicles"]["vehicle_type"].value_counts())

INSURAFLOW RAW DATA SUMMARY
Customers        : 500
Vehicles         : 500
Policies         : 750
Claims           : 300
Premium Payments : 1,211
Claim Payments   : 68

Policy Status:
policy_status
Active       341
Expired      275
Pending       56
Cancelled     49
Lapsed        29
Name: count, dtype: int64

Claim Status:
claim_status
Approved        79
Settled         76
Under Review    72
Submitted       43
Rejected        30
Name: count, dtype: int64

Claim Type:
claim_type
Third Party Damage    72
Natural Disaster      68
Theft                 65
Fire                  48
Accident              47
Name: count, dtype: int64

Vehicle Type:
vehicle_type
SUV          239
Hatchback    127
Sedan         74
MPV           60
Name: count, dtype: int64
